# Evaluierung & Analyse des Similarity-Threshold-Experiments (0.60 vs. 0.70 vs. 0.80 bis 0.98)

Dieses Notebook analysiert systematisch den Einfluss des **minimalen semantischen Ähnlichkeits-Schwellenwerts** ($s_{\min} \in \{0.60, 0.70, 0.80\}$ bei $s_{\max} = 0.98$) auf die Datenqualität, das Metrik-Training und das generative Übersetzungsmodell:

1. **Datenbasis & Korpusebene:** Analyse des Trade-offs zwischen Datenvolumen (Sample-Anzahl, Token-Volumen) und semantischer Ausrichtung.
2. **Continuous MixUp Regressor (BiLSTM):** In-Domain Regressionsgüte ($MSE, MAE, R^2, r$) vs. Out-of-Domain Trennschärfe (ROC-AUC & Perfect Pair Match Rate auf dem *Lebenshilfe*-Benchmark).
3. **SFT Übersetzungsmodell (mBART-50 LoRA):** In-Domain Konvergenz vs. Out-of-Domain Generierungsqualität (Stilistische Einfachheit $R_{\text{style}}$, semantischer Erhalt $R_{\text{sem}}$, Referenztreue $Sim_{\text{ref}}$ und Composite Reward).

---

In [ ]:
import os
import sys
import json
import glob
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'data')) and os.path.exists(os.path.join(p, 'results')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser('~/Documents/Master Thesis'))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Arbeitsverzeichnis:', os.getcwd())
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 150


## 1. Datenbasis & Korpuseigenschaften über Schwellenwerte

Untersuchung der Datenretention im bereinigten Master-Korpus (`data/analysis/corpus_master.csv`).

In [ ]:
CORPUS_CSV = os.path.join(REPO_ROOT, 'data/analysis/corpus_master.csv')
df_corpus = pd.read_csv(CORPUS_CSV)

threshold_stats = []
for s_min in [0.60, 0.70, 0.80, 0.85, 0.90]:
    sub = df_corpus[(df_corpus['semantic_similarity_8192'] >= s_min) & (df_corpus['semantic_similarity_8192'] <= 0.98)]
    threshold_stats.append({
        'Schwellenwert': f'[{s_min:.2f}, 0.98]',
        's_min': s_min,
        'Artikelpaare': len(sub),
        'Retention (%)': round((len(sub) / len(df_corpus)) * 100.0, 2),
        'AS Tokens': int(sub['as_tokens'].sum()),
        'LS Tokens': int(sub['ls_tokens'].sum()),
        'Token-Ratio (LS/AS)': round(sub['ls_tokens'].sum() / max(1, sub['as_tokens'].sum()), 3),
        'NER Recall AS->LS': round(sub['ner_recall_as_ls'].mean(), 4) if 'ner_recall_as_ls' in sub.columns else np.nan,
        'Ø Flesch LS': round(sub['ls_flesch'].mean(), 2) if 'ls_flesch' in sub.columns else np.nan,
    })

df_tstats = pd.DataFrame(threshold_stats)
display(df_tstats)


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
color = '#2b5c8f'
ax1.set_xlabel('Minimaler Ähnlichkeits-Schwellenwert ($s_{min}$)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Artikelpaare ($N$)', color=color, fontsize=12, fontweight='bold')
ax1.plot(df_tstats['s_min'], df_tstats['Artikelpaare'], marker='o', color=color, linewidth=2.5, label='Artikelpaare')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = '#d95f02'
ax2.set_ylabel('Erhalt des Korpus (%)', color=color, fontsize=12, fontweight='bold')
ax2.plot(df_tstats['s_min'], df_tstats['Retention (%)'], marker='s', color=color, linestyle='--', linewidth=2, label='Retention (%)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Datenretention und Sample-Verlust über die Ähnlichkeits-Schwellenwerte', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()


## 2. Evaluationsergebnisse laden (MixUp & SFT)

Laden der konsolidierten Zusammenfassungs- und Detaildateien aus `results/experiments/similarity_threshold/`.

In [ ]:
SUMMARY_CSV = os.path.join(REPO_ROOT, 'results/experiments/similarity_threshold/similarity_threshold_summary.csv')
DETAILS_CSV = os.path.join(REPO_ROOT, 'results/experiments/similarity_threshold/similarity_threshold_details.csv')

if os.path.exists(SUMMARY_CSV):
    df_summary = pd.read_csv(SUMMARY_CSV)
    display(Markdown('### Aggregierte Metriken:'))
    display(df_summary)
else:
    print(f'Hinweis: {SUMMARY_CSV} noch nicht generiert. Führe zuerst das Trainings- und Evaluationsskript aus.')
    df_summary = pd.DataFrame()


## 3. MixUp Regressor: In-Domain vs. Out-of-Domain Generalisierung

Gegenüberstellung der Regressionsgüte auf dem Testset vs. Trennschärfe auf dem ungesehenen *Lebenshilfe*-Datensatz.

In [ ]:
if not df_summary.empty and 'MixUp Regressor' in df_summary.get('model_type', '').values:
    df_m = df_summary[df_summary['model_type'] == 'MixUp Regressor'].sort_values(by='min_sim')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Subplot 1: In-Domain Fehlermaße
    ax1.plot(df_m['min_sim'], df_m['in_domain_test_mse'], marker='o', color='#1b9e77', linewidth=2, label='Test MSE')
    ax1.plot(df_m['min_sim'], df_m['in_domain_test_mae'], marker='s', color='#d95f02', linewidth=2, label='Test MAE')
    ax1.set_xlabel('Schwellenwert $s_{min}$', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Fehler (niedriger = besser)', fontsize=12, fontweight='bold')
    ax1.set_title('In-Domain Test-Fehler (MixUp)', fontsize=13, fontweight='bold')
    ax1.legend()
    
    # Subplot 2: OOD Lebenshilfe Trennschärfe
    if 'ood_separation_auc' in df_m.columns:
        ax2.plot(df_m['min_sim'], df_m['ood_separation_auc'], marker='^', color='#7570b3', linewidth=2.5, label='OOD Separation AUC')
    if 'ood_perfect_pair_match_pct' in df_m.columns:
        ax2.plot(df_m['min_sim'], df_m['ood_perfect_pair_match_pct'] / 100.0, marker='D', color='#e7298a', linewidth=2, linestyle='--', label='Perfect Pair Match')
    ax2.set_xlabel('Schwellenwert $s_{min}$', fontsize=12, fontweight='bold')
    ax2.set_ylabel('OOD Güte [0.0 - 1.0]', fontsize=12, fontweight='bold')
    ax2.set_title('Out-of-Domain Lebenshilfe Güte', fontsize=13, fontweight='bold')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()


## 4. SFT Übersetzungsmodell: Stil, Semantik & Referenztreue

Analyse der generierten Übersetzungen auf dem unabhängigen *Lebenshilfe*-Datensatz.

In [ ]:
if not df_summary.empty and 'SFT mBART-50 LoRA' in df_summary.get('model_type', '').values:
    df_s = df_summary[df_summary['model_type'] == 'SFT mBART-50 LoRA'].sort_values(by='min_sim')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Subplot 1: Reward-Komponenten
    ax1.plot(df_s['min_sim'], df_s['r_style_mean'], marker='o', color='#e7298a', linewidth=2.5, label='Einfachheit ($R_{style}$)')
    ax1.plot(df_s['min_sim'], df_s['r_sem_as_mean'], marker='s', color='#1f78b4', linewidth=2.5, label='Semantik zu AS ($R_{sem}$)')
    ax1.plot(df_s['min_sim'], df_s['composite_reward_mean'], marker='D', color='#33a02c', linewidth=2.5, linestyle='--', label='Composite Reward')
    ax1.set_xlabel('Schwellenwert $s_{min}$', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Score / Reward', fontsize=12, fontweight='bold')
    ax1.set_title('SFT Reward-Metriken (Lebenshilfe)', fontsize=13, fontweight='bold')
    ax1.legend()
    
    # Subplot 2: Lexikalische Metriken & Truncation Rate
    ax2.plot(df_s['min_sim'], df_s['bleu_mean'], marker='o', color='#ff7f00', linewidth=2, label='BLEU')
    ax2.plot(df_s['min_sim'], df_s['rouge_l_mean'], marker='s', color='#6a3d9a', linewidth=2, label='ROUGE-L')
    ax2.set_xlabel('Schwellenwert $s_{min}$', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Lexikalische Überlappung', fontsize=12, fontweight='bold')
    ax2.set_title('SFT Lexikalische Scores (Lebenshilfe)', fontsize=13, fontweight='bold')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()


## 5. LaTeX Tabellen Generator für die Masterarbeit

Generiert formatierte Tabellen für Kapitel 3 (*Materials*) und Kapitel 5 (*Experiments & Results*).

In [ ]:
print('% ==============================================================================')
print('% LaTeX Tabelle: Similarity Threshold Experiment Zusammenfassung')
print('% ==============================================================================')
print('\\begin{table}[htbp]')
print('\\centering\\small')
print('\\caption{Vergleich der Modellgüte in Abhängigkeit des Ähnlichkeits-Schwellenwerts $s_{\\min}$.}')
print('\\label{tab:similarity_threshold_results}')
print('\\begin{tabular}{@{}llrrrrr@{}}')
print('\\toprule')
print('\\textbf{Modell} & \\textbf{Filter} & \\textbf{In-Domain} & \\textbf{OOD AUC / Treue} & \\textbf{Stil ($R_{style}$)} & \\textbf{Semantik ($R_{sem}$)} & \\textbf{Composite} \\\\')
print('\\midrule')

if not df_summary.empty:
    for _, r in df_summary.iterrows():
        m_name = r.get('model_type', 'Modell')
        filt = f"${r.get('min_sim', 0):.2f} \\le s \\le {r.get('max_sim', 0):.2f}$"
        if 'MixUp' in m_name:
            in_d = f"MSE {r.get('in_domain_test_mse', 0):.4f}"
            ood = f"{r.get('ood_separation_auc', 0):.4f} (AUC)"
            r_st = f"{r.get('ood_mean_score_ls', 0):.4f}"
            r_se = f"{r.get('ood_mean_score_as', 0):.4f}"
            comp = f"$\\Delta$ {r.get('ood_score_delta', 0):.4f}"
        else:
            in_d = f"Loss {r.get('best_val_loss', 0):.4f}"
            ood = f"{r.get('sim_ref_mean', 0):.4f} (Sim)"
            r_st = f"{r.get('r_style_mean', 0):.4f}"
            r_se = f"{r.get('r_sem_as_mean', 0):.4f}"
            comp = f"{r.get('composite_reward_mean', 0):.4f}"
        print(f"{m_name} & {filt} & {in_d} & {ood} & {r_st} & {r_se} & {comp} \\\\")

print('\\bottomrule')
print('\\end{tabular}')
print('\\end{table}')
